# C3 – Day 1: Hirsch Dataset Exploration
**AnchorAI | Component 3: Cognitive Checkpointing**

This notebook explores the **Hirsch et al. resumption costs** dataset before any modelling.
We inspect column meanings, participants, conditions, and the distribution of **resumption RT** (our target variable).

> Run on **CPU runtime** in Google Colab. No GPU needed today.

## Step 0 – Mount Google Drive and set paths
Upload the three CSV files to `MyDrive/C3/data/raw/hirsch/` in your Google Drive before running.

In [ ]:
import os, csv, io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Mount Drive (skip if running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/C3/data/raw/hirsch'
    print('Colab – Drive mounted.')
except ImportError:
    DATA_DIR = r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing\data\raw\hirsch'
    print('Running locally.')

print('Data directory:', DATA_DIR)
print('Files:', os.listdir(DATA_DIR))

## Step 1 – Load all three experiment files

The dataset comes in three files (Exp1, Exp2, Exp3). Each file uses a special quoting format where the entire row is wrapped in double-quotes. The helper function below handles that.

In [ ]:
def load_hirsch_csv(filepath):
    """Read a Hirsch CSV file that wraps every row in outer double-quotes."""
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()

    def parse_line(line):
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1]           # strip outer quotes
            line = line.replace('""', '"')  # unescape inner quotes
        return next(csv.reader([line]))

    header = parse_line(raw_lines[0])
    rows   = [parse_line(l) for l in raw_lines[1:] if l.strip()]
    return pd.DataFrame(rows, columns=header)


# Load and tag each experiment
exp_files = {
    'Exp1': os.path.join(DATA_DIR, 'raw-data_Exp1.csv'),
    'Exp2': os.path.join(DATA_DIR, 'raw-data_Exp2.csv'),
    'Exp3': os.path.join(DATA_DIR, 'raw-data_Exp3.csv'),
}

frames = []
for exp_name, path in exp_files.items():
    tmp = load_hirsch_csv(path)
    tmp['experiment'] = exp_name
    frames.append(tmp)
    print(f'{exp_name}: {tmp.shape[0]:,} rows, {tmp.shape[1]} cols')

df = pd.concat(frames, ignore_index=True)
print(f'\nCombined shape: {df.shape}')
df.head()

## Step 2 – Column overview
Let us see the column names and data types. All values load as strings first; we convert numeric columns below.

In [ ]:
print('Columns:', df.columns.tolist())
print()
print(df.dtypes)

In [ ]:
# Convert numeric columns (RT, error, trial, etc.)
numeric_cols = ['pp', 'bBlock', 'blocktype', 'trial', 'trialtype', 'task',
                'sequence', 'RT', 'error', 'Stimulus1', 'Stimulus2', 'Stimulus3', 'Errortyp']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('After conversion:')
print(df.dtypes)
print()
df.describe()

## Step 3 – Participants
Column `pp` is the participant ID. We **must never split by row randomly** – we split by participant to avoid data leakage (a participant's data must not appear in both train and test).

In [ ]:
participants = df['pp'].dropna().unique()
print(f'Total unique participants: {len(participants)}')
print(f'Participant IDs (first 20): {sorted(participants)[:20]}')
print()
print('Trials per participant (top 10):')
print(df['pp'].value_counts().head(10))

## Step 4 – Experimental conditions
What values do `blocktype`, `task`, `trialtype` take? These tell us about task and interruption conditions.

In [ ]:
for col in ['blocktype', 'task', 'trialtype', 'sequence']:
    if col in df.columns:
        print(f'{col} values: {sorted(df[col].dropna().unique())}')

## Step 5 – Missing values
Check for missing or NaN values before training.

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values!')
print(f'\nTotal missing cells: {missing.sum()}')

## Step 6 – Resumption RT distribution
`RT` is our target variable (resumption reaction time in ms). We will label trials as **Low / Medium / High** difficulty using terciles.
- Low  = bottom 33% of RT (quick resumption = easy)
- Medium = middle 33%
- High = top 33% of RT (slow resumption = difficult)

In [ ]:
rt = df['RT'].dropna()

t33 = rt.quantile(0.333)
t67 = rt.quantile(0.667)
print(f'RT range       : {rt.min():.0f} – {rt.max():.0f} ms')
print(f'Mean RT        : {rt.mean():.0f} ms')
print(f'Tercile cuts   : Low < {t33:.0f} ms | Medium {t33:.0f}–{t67:.0f} ms | High > {t67:.0f} ms')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with tercile lines
axes[0].hist(rt, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(t33, color='orange', linestyle='--', linewidth=2, label=f'33rd pct ({t33:.0f} ms)')
axes[0].axvline(t67, color='red',    linestyle='--', linewidth=2, label=f'67th pct ({t67:.0f} ms)')
axes[0].set_title('Resumption RT Distribution (all experiments)')
axes[0].set_xlabel('Reaction Time (ms)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot per experiment
data_by_exp = [df[df['experiment'] == e]['RT'].dropna() for e in ['Exp1', 'Exp2', 'Exp3']]
axes[1].boxplot(data_by_exp, labels=['Exp1', 'Exp2', 'Exp3'])
axes[1].set_title('RT by Experiment')
axes[1].set_xlabel('Experiment')
axes[1].set_ylabel('Reaction Time (ms)')

plt.tight_layout()
plt.savefig('hirsch_rt_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## Step 7 – Assign Low / Medium / High labels

In [ ]:
def assign_label(rt_val):
    if pd.isna(rt_val):
        return np.nan
    if rt_val <= t33:
        return 'Low'
    elif rt_val <= t67:
        return 'Medium'
    else:
        return 'High'

df['difficulty'] = df['RT'].apply(assign_label)

print('Label distribution:')
print(df['difficulty'].value_counts())
print()
df[['pp', 'experiment', 'RT', 'difficulty']].head(10)

## Summary – What we found today
Fill in after running:

| Metric | Value |
|--------|-------|
| Total rows | ? |
| Unique participants | ? |
| Missing RT values | ? |
| RT range | ? ms – ? ms |
| Tercile cuts (Low/Med/High) | ? / ? ms |
| Label balance | Low: ? \| Med: ? \| High: ? |

**These numbers go into the PP1 slide 'Dataset'.**

→ Next: **Day 2 – Preprocessing and participant-wise split**